In [1]:
import pandas as pd

df = pd.read_csv("../data/email_evaluation_dataset_arbind.csv")
df.head()


,email_text,expected_action,expected_tone
0,Reminder: Team meeting scheduled tomorrow at 1...,respond,polite
1,Your electricity bill payment is overdue. Plea...,notify,urgent
2,Congratulations! You have won a gift voucher. ...,ignore,neutral
3,Please find attached the internship offer lett...,respond,polite
4,System alert: Multiple failed login attempts d...,notify,urgent


In [2]:
def classify_email(row):
    action = str(row['expected_action']).lower()
    tone = str(row['expected_tone']).lower()
    
    if action == "notify" or tone == "urgent":
        return "Important: Notify immediately"
    elif action == "ignore" or tone == "polite":
        return "Polite: No action needed"
    else:
        return "Normal: Needs a response"

df['Classification'] = df.apply(classify_email, axis=1)
df.head()


,email_text,expected_action,expected_tone,Classification
0,Reminder: Team meeting scheduled tomorrow at 1...,respond,polite,Polite: No action needed
1,Your electricity bill payment is overdue. Plea...,notify,urgent,Important: Notify immediately
2,Congratulations! You have won a gift voucher. ...,ignore,neutral,Polite: No action needed
3,Please find attached the internship offer lett...,respond,polite,Polite: No action needed
4,System alert: Multiple failed login attempts d...,notify,urgent,Important: Notify immediately


In [3]:
def email_assistant(email_text):
    text = email_text.lower()

    # Urgent / notify rules
    if any(word in text for word in ["urgent", "deadline", "submit", "overdue", "immediately", "alert"]):
        return "notify", "urgent"

    # Ignore rules
    elif any(word in text for word in ["thank you", "newsletter", "promotion", "sale", "congratulations"]):
        return "ignore", "neutral"

    # Default: normal emails
    else:
        return "respond", "polite"


In [4]:
import pandas as pd

df = pd.read_csv("../data/email_evaluation_dataset_arbind.csv")
print("Emails loaded:", len(df))
df.head()


Emails loaded: 98


,email_text,expected_action,expected_tone
0,Reminder: Team meeting scheduled tomorrow at 1...,respond,polite
1,Your electricity bill payment is overdue. Plea...,notify,urgent
2,Congratulations! You have won a gift voucher. ...,ignore,neutral
3,Please find attached the internship offer lett...,respond,polite
4,System alert: Multiple failed login attempts d...,notify,urgent


In [5]:
df["assistant_output"] = df["email_text"].apply(email_assistant)


In [6]:
df[["predicted_action", "predicted_tone"]] = pd.DataFrame(
    df["assistant_output"].tolist(),
    index=df.index
)


In [7]:
df["action_correct"] = df["predicted_action"] == df["expected_action"]
df["tone_correct"] = df["predicted_tone"] == df["expected_tone"]


In [8]:
action_accuracy = df["action_correct"].mean() * 100
tone_accuracy = df["tone_correct"].mean() * 100

print(f"Action Accuracy: {action_accuracy:.2f}%")
print(f"Tone Accuracy: {tone_accuracy:.2f}%")


Action Accuracy: 70.41%
Tone Accuracy: 58.16%


In [9]:
errors = df[df["action_correct"] == False][
    ["email_text", "expected_action", "predicted_action"]
]

errors.head(10)


,email_text,expected_action,predicted_action
8,Security notice: Password change required with...,notify,respond
11,Your subscription for streaming service expire...,notify,respond
12,Reminder: Submit your expense report by end of...,respond,notify
13,Holiday greeting from CEO: Wishing everyone a ...,ignore,respond
15,Your password will expire in 3 days. Change it...,notify,respond
21,System maintenance scheduled tonight from 12 A...,notify,respond
22,Congratulations! You have been shortlisted for...,respond,ignore
23,Your credit card statement is ready for download.,notify,respond
27,Reminder: Submit project proposal by Monday noon.,respond,notify
28,Your package has been delayed due to weather c...,notify,respond


In [10]:
output_path = "../data/milestone2_output_arbind.csv"
df.to_csv(output_path, index=False)

print("Milestone 2 output saved at:", output_path)


Milestone 2 output saved at: ../data/milestone2_output_arbind.csv


In [11]:
df.head()

,email_text,expected_action,expected_tone,assistant_output,predicted_action,predicted_tone,action_correct,tone_correct
0,Reminder: Team meeting scheduled tomorrow at 1...,respond,polite,"(respond, polite)",respond,polite,True,True
1,Your electricity bill payment is overdue. Plea...,notify,urgent,"(notify, urgent)",notify,urgent,True,True
2,Congratulations! You have won a gift voucher. ...,ignore,neutral,"(ignore, neutral)",ignore,neutral,True,True
3,Please find attached the internship offer lett...,respond,polite,"(respond, polite)",respond,polite,True,True
4,System alert: Multiple failed login attempts d...,notify,urgent,"(notify, urgent)",notify,urgent,True,True


In [12]:
from langsmith import Client

client = Client()
print("LangSmith client initialized")



LangSmith client initialized


In [13]:
#DEFINE THE JUDGE PROMPT

judge_prompt='''You are an evaluator . Compare the model output with ideal answer

Check:
1.Action correctness
2.Tone correctness 


Give Score :
1=correct
0=incorrect

'''



In [14]:
def evaluate(agent_output,ideal_action,ideal_tone):
    if(
        agent_output["action"]==ideal_action
        and agent_output["tone"]==ideal_tone
    ):
        return 1
    else :
        return 0

In [15]:
agent_output={
    "action" : "notify",
    "tone" : "urgent"
}

In [16]:
ideal_action="notify"
ideal_tone="urgent"

In [17]:
df.head()

,email_text,expected_action,expected_tone,assistant_output,predicted_action,predicted_tone,action_correct,tone_correct
0,Reminder: Team meeting scheduled tomorrow at 1...,respond,polite,"(respond, polite)",respond,polite,True,True
1,Your electricity bill payment is overdue. Plea...,notify,urgent,"(notify, urgent)",notify,urgent,True,True
2,Congratulations! You have won a gift voucher. ...,ignore,neutral,"(ignore, neutral)",ignore,neutral,True,True
3,Please find attached the internship offer lett...,respond,polite,"(respond, polite)",respond,polite,True,True
4,System alert: Multiple failed login attempts d...,notify,urgent,"(notify, urgent)",notify,urgent,True,True


In [18]:
score=evaluate(agent_output,ideal_action,ideal_tone)
print(score)

1
